# Module 42: Building a RAG Pipeline with PyTorch

This notebook walks through building a Retrieval-Augmented Generation (RAG) pipeline
from scratch using PyTorch. We'll implement:

1. Document encoding with sentence transformers
2. A pure-PyTorch vector index
3. Document chunking strategies
4. Retrieval evaluation metrics
5. End-to-end RAG query pipeline

In [ ]:
import torch
import torch.nn.functional as F
import math

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Vector Index (Pure PyTorch)

We start with the simplest possible vector store — normalized embeddings
with cosine similarity computed via matrix multiplication.

In [ ]:
class VectorIndex:
    """In-memory vector store using cosine similarity."""

    def __init__(self):
        self.embeddings: torch.Tensor | None = None
        self.documents: list[str] = []

    def add(self, documents: list[str], embeddings: torch.Tensor):
        embeddings = F.normalize(embeddings, p=2, dim=1)
        if self.embeddings is None:
            self.embeddings = embeddings
        else:
            self.embeddings = torch.cat([self.embeddings, embeddings], dim=0)
        self.documents.extend(documents)

    def search(self, query_embedding: torch.Tensor, top_k: int = 5):
        query_embedding = F.normalize(query_embedding, p=2, dim=1)
        similarities = torch.mm(query_embedding, self.embeddings.T).squeeze(0)
        k = min(top_k, len(self.documents))
        scores, indices = torch.topk(similarities, k)
        return [(self.documents[idx], scores[i].item()) for i, idx in enumerate(indices)]

    def __len__(self):
        return len(self.documents)


# Demo with random embeddings
torch.manual_seed(42)
index = VectorIndex()

docs = [
    "PyTorch provides tensor computation with GPU acceleration",
    "torch.compile uses TorchDynamo and TorchInductor for optimization",
    "FSDP2 shards model parameters across data-parallel workers",
    "FlexAttention allows custom attention patterns via score_mod",
    "Mixed precision uses FP16/BF16 for 2-3x speedup",
]

# Simulate embeddings (in practice, use a sentence transformer)
fake_embeddings = torch.randn(len(docs), 384)
index.add(docs, fake_embeddings)

# Query
query_emb = fake_embeddings[1:2] + 0.1 * torch.randn(1, 384)  # Similar to doc[1]
results = index.search(query_emb, top_k=3)

print("Query: something about torch.compile")
print("\nTop-3 results:")
for doc, score in results:
    print(f"  [{score:.3f}] {doc}")

## 2. Document Chunking

How you split documents dramatically affects retrieval quality.
Let's compare different strategies.

In [ ]:
import re

sample_text = """PyTorch is an open-source machine learning framework. It was developed by Meta AI
and released in 2016. PyTorch provides two high-level features: tensor computation with GPU
acceleration, and deep neural networks built on a tape-based autograd system.

The framework has become the most popular choice for research. It offers dynamic computation
graphs, which allow flexible model architectures. Researchers can use standard Python control
flow in their models.

torch.compile was introduced in PyTorch 2.0. It uses TorchDynamo to capture Python bytecode
into an FX graph. The graph is then optimized by TorchInductor, which generates fast kernels
for both CPU and GPU."""


def chunk_by_sentences(text, sentences_per_chunk=3, overlap=1):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    chunks = []
    start = 0
    while start < len(sentences):
        end = start + sentences_per_chunk
        chunk = ' '.join(sentences[start:end])
        if chunk:
            chunks.append(chunk)
        start += sentences_per_chunk - overlap
    return chunks


def chunk_by_paragraphs(text, max_size=500):
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    chunks, current = [], ''
    for para in paragraphs:
        if len(current) + len(para) + 2 <= max_size:
            current = f'{current}\n\n{para}' if current else para
        else:
            if current:
                chunks.append(current)
            current = para
    if current:
        chunks.append(current)
    return chunks


# Compare strategies
print("=" * 50)
print("Sentence-based (3 sentences, overlap=1):")
for i, chunk in enumerate(chunk_by_sentences(sample_text)):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:70]}...")

print("\nParagraph-based (max 300 chars):")
for i, chunk in enumerate(chunk_by_paragraphs(sample_text, max_size=300)):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:70]}...")

## 3. Retrieval Evaluation Metrics

Standard IR metrics to measure how well your retrieval works.

In [ ]:
def recall_at_k(retrieved, relevant, k):
    """What fraction of relevant docs appear in top-k?"""
    return len(set(retrieved[:k]) & set(relevant)) / len(relevant) if relevant else 0.0


def precision_at_k(retrieved, relevant, k):
    """What fraction of top-k are relevant?"""
    return len(set(retrieved[:k]) & set(relevant)) / k if k > 0 else 0.0


def mrr(retrieved, relevant):
    """Reciprocal rank of first relevant result."""
    for i, doc in enumerate(retrieved):
        if doc in set(relevant):
            return 1.0 / (i + 1)
    return 0.0


def ndcg_at_k(retrieved, relevant, k):
    """Normalized DCG — accounts for position of relevant results."""
    relevant_set = set(relevant)
    dcg = sum(1.0 / math.log2(i + 2) for i, d in enumerate(retrieved[:k]) if d in relevant_set)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0


# Example
retrieved = ["doc_1", "doc_3", "doc_5", "doc_2", "doc_7"]
relevant = ["doc_1", "doc_2", "doc_4"]

print(f"Retrieved: {retrieved}")
print(f"Relevant:  {relevant}")
print()

for k in [1, 3, 5]:
    print(f"  @{k}: Recall={recall_at_k(retrieved, relevant, k):.3f}, "
          f"Precision={precision_at_k(retrieved, relevant, k):.3f}, "
          f"NDCG={ndcg_at_k(retrieved, relevant, k):.3f}")

print(f"  MRR: {mrr(retrieved, relevant):.3f}")

## 4. Embedding Similarity Analysis

Visualize how well embeddings capture semantic similarity.

In [ ]:
# Simulate semantic embeddings where similar docs have similar vectors
torch.manual_seed(42)

# Create 3 clusters of documents
cluster_centers = torch.randn(3, 128)
cluster_labels = ["pytorch_basics", "compilation", "distributed"]

all_embeddings = []
all_labels = []
for i, (center, label) in enumerate(zip(cluster_centers, cluster_labels)):
    noise = 0.3 * torch.randn(5, 128)
    cluster_embs = center.unsqueeze(0) + noise
    all_embeddings.append(cluster_embs)
    all_labels.extend([label] * 5)

embeddings = F.normalize(torch.cat(all_embeddings, dim=0), p=2, dim=1)

# Compute pairwise similarities
sim_matrix = torch.mm(embeddings, embeddings.T)

print("Pairwise Similarity Matrix (15x15):")
print(f"  Shape: {sim_matrix.shape}")
print(f"  Within-cluster avg similarity: ", end="")

within_sims = []
between_sims = []
for i in range(15):
    for j in range(i+1, 15):
        if all_labels[i] == all_labels[j]:
            within_sims.append(sim_matrix[i, j].item())
        else:
            between_sims.append(sim_matrix[i, j].item())

print(f"{sum(within_sims)/len(within_sims):.3f}")
print(f"  Between-cluster avg similarity: {sum(between_sims)/len(between_sims):.3f}")
print(f"  Separation ratio: {(sum(within_sims)/len(within_sims)) / (sum(between_sims)/len(between_sims)):.2f}x")

## 5. Context Assembly & Prompt Engineering

How to format retrieved documents into a prompt for the generator.

In [ ]:
def build_rag_prompt(query, retrieved_docs, max_context_chars=2000):
    """Assemble retrieved docs into a generation prompt."""
    context_parts = []
    total_chars = 0
    for doc, score in retrieved_docs:
        if total_chars + len(doc) > max_context_chars:
            break
        context_parts.append(f"[{score:.3f}] {doc}")
        total_chars += len(doc)

    context = "\n\n".join(context_parts)
    return f"""Answer the question using only the provided context.
If the answer isn't in the context, say "I don't know."

Context:
{context}

Question: {query}
Answer:"""


# Example
query = "How does torch.compile optimize models?"
retrieved = [
    ("torch.compile uses TorchDynamo to capture Python bytecode into FX graphs, "
     "then TorchInductor generates optimized kernels.", 0.92),
    ("TorchInductor supports both CPU (via C++/OpenMP) and GPU (via Triton) "
     "code generation.", 0.85),
    ("Graph breaks occur when Dynamo encounters unsupported Python constructs.", 0.78),
]

prompt = build_rag_prompt(query, retrieved)
print(prompt)

## 6. Performance: Scaling the Index

Benchmark retrieval speed as index size grows.

In [ ]:
import time

torch.manual_seed(0)
dim = 384

print(f"{'Index Size':<12} {'Search Time (ms)':<18} {'Throughput (q/s)'}")
print("-" * 50)

for n_docs in [100, 1_000, 10_000, 50_000, 100_000]:
    embeddings = F.normalize(torch.randn(n_docs, dim), p=2, dim=1)
    query = F.normalize(torch.randn(1, dim), p=2, dim=1)

    # Warmup
    _ = torch.mm(query, embeddings.T)

    # Benchmark
    n_queries = 100
    start = time.perf_counter()
    for _ in range(n_queries):
        sims = torch.mm(query, embeddings.T)
        _ = torch.topk(sims, 5)
    elapsed = time.perf_counter() - start

    ms_per_query = (elapsed / n_queries) * 1000
    throughput = n_queries / elapsed
    print(f"{n_docs:<12,} {ms_per_query:<18.3f} {throughput:,.0f}")

## Key Takeaways

1. **RAG = Retrieve + Generate**: Combines search with LLM generation for factual answers
2. **Embedding quality is critical**: The encoder determines retrieval quality
3. **Chunking matters**: Too small = lost context, too large = noise in retrieval
4. **Pure PyTorch scales**: Cosine similarity via matmul handles 100K+ docs at <1ms/query on CPU
5. **Evaluate both retrieval AND generation**: Use recall/NDCG for retrieval, faithfulness for generation